## Homework 1.5 - Coding

This is the coding portion of the homework assignment for Section 1.5.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import time


## Problem 1.26

### Part (i)

Create a function `create_arrays_prob_26()`, which accepts as an argument $k$, and returns as numpy arrays:

1.  $A$: A random matrix of size $2^k \times 2^k$
2.  $B$: A random matrix of size $2^k \times 2^k$
3.  $\mathbf{x}$: A random vector of size $2^k$.

In [ ]:
def create_arrays_prob_26(k: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Creates the arrays called for in Problem 1.26.
    
    Args:
        k (int): The power of 2 used for dimensions of the arrays
    
    Returns:
        A (np.ndarray): A random matrix of size $2^k \times 2^k$
        B (np.ndarray): A random matrix of size $2^k \times 2^k$
        x (np.ndarray): A random vector of size 2^k
    """
    n = 2 ** k

    A = np.random.random((n, n))   # random n x n matrix
    B = np.random.random((n, n))   # random n x n matrix
    x = np.random.random(n)        # random vector of length n

    return A, B, x


### Part (ii)

For $k = 1, 2, \ldots, 11$, time the computation of $(AB)\mathbf{x}$ versus the computation of $A(B\mathbf{x})$.

Store the times in the lists `matrices_first_times` and `vector_first_times`, respectively.

In [ ]:
matrices_first_times = []
vector_first_times = []
k_vals = np.arange(1,12)

for k in k_vals:
    # Create the arrays BEFORE starting any timers
    A, B, x = create_arrays_prob_26(k)

    # Time (AB)x: matrix-matrix product first, then matrix-vector
    start = time.time_ns()
    (A @ B) @ x
    end = time.time_ns()
    matrices_first_times.append(end - start)

    # Time A(Bx): two matrix-vector products
    start = time.time_ns()
    A @ (B @ x)
    end = time.time_ns()
    vector_first_times.append(end - start)


For each $k$, find the ratio of the time it takes to compute $(AB)\mathbf{x}$ versus $A(B\mathbf{x})$. Store the result in the list `ratios`.

In [ ]:
ratios = []

# ratio = time for (AB)x divided by time for A(Bx), one entry per k
for mat_time, vec_time in zip(matrices_first_times, vector_first_times):
    ratios.append(mat_time / vec_time)

ratios


Once you have done the above work, run the next cell to see how the ratios compare over time:

In [ ]:
plt.plot(k_vals, ratios)

When $k$ increases by one, does the ratio of the times of the two computations change? By how much? Explain this in terms of what we ahve discussed about the complexity of matrix-matrix and matrix-vector multiplication.

**Explanation**: Yes, the ratio doubles each time $k$ goes up by one.

With $n = 2^k$, a matrix-matrix product costs about $2n^3$ flops and a matrix-vector product costs about $2n^2$.

- $(AB)\mathbf{x}$: about $2n^3$ flops
- $A(B\mathbf{x})$: two matrix-vector products, about $4n^2$ flops

The ratio of the leading terms is
$$\frac{2n^3}{4n^2} = \frac{n}{2} = 2^{k-1},$$

which doubles when $k$ increases by one. So the graph curves upward instead of staying flat. Both groupings give the same answer, but doing the matrices first does about $n/2$ times more work.

The small values of $k$ are just noise, since those computations are too fast for the clock to measure. The measured ratio is also lower than predicted, because NumPy runs matrix-matrix products on a fast multithreaded routine. The growth still matches.

---

## Problem 1.27

**IMPORTANT:** You should have verified algebraically (by hand) that
$$(I_n + \mathbf{u} \mathbf{v}^\top) \mathbf{x} = \mathbf{x} + \mathbf{u}\left(\mathbf{v}^\top \mathbf{x} \right) \text{ for any } \mathbf{u}, \mathbf{v}, \mathbf{x} \in \mathbb{R}^n$$
and turned it into the written assignment for 1.5. If you have not done that, do it now before proceeding.

### Part (i)

Write a function `create_arrays_prob_27()` which accepts as an argument $n$, and returns the following as numpy arrays:

1. The $2^n \times 2^n$ identity matrix $I$
2. $\mathbf{u}$ (a random vector of size $2^n$)
3. $\mathbf{v}$ (a random vector of size $2^n$)
4. $\mathbf{x}$ (a random vector of size $2^n$)

In [ ]:
def create_arrays_prob_27(n: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Creates the arrays called for in Problem 1.27.
    
    Args:
        n (int): The power of 2 used for dimensions of the arrays
    
    Returns:
        I (np.ndarray): The 2^n x 2^n identity matrix
        u (np.ndarray): A random vector of size 2^n
        v (np.ndarray): A random vector of size 2^n
        x (np.ndarray): A random vector of size 2^n
    """
    size = 2 ** n

    I = np.eye(size)                # size x size identity matrix
    u = np.random.random(size)      # random vector of length size
    v = np.random.random(size)      # random vector of length size
    x = np.random.random(size)      # random vector of length size

    return I, u, v, x


Time the computation of $(I + \mathbf{u}\mathbf{v}^\top)\mathbf{x}$ versus $\mathbf{x} + \mathbf{u}(\mathbf{v}^\top \mathbf{x})$.

Store the results in the lists `compact_times` and `expanded_times`, respectively.

In [ ]:
n_vals = np.arange(1,12)
compact_times = []      # Times for (I + uv^T)x
expanded_times = []     # Times for x + u(v^T x)

for n in n_vals:
    # Create the arrays BEFORE starting any timers
    I, u, v, x = create_arrays_prob_27(n)

    # Time (I + uv^T)x: build the full matrix, then multiply by x
    start = time.time_ns()
    (I + np.outer(u, v)) @ x
    end = time.time_ns()
    compact_times.append(end - start)

    # Time x + u(v^T x): a dot product, a scalar-vector product, and a vector sum
    start = time.time_ns()
    x + u * (v @ x)
    end = time.time_ns()
    expanded_times.append(end - start)


For each $k$, find the ratio of the time it takes to compute $(I + \mathbf{u} \mathbf{v}^\top)\mathbf{x}$ versus $\mathbf{x} + \mathbf{u}(\mathbf{v}^\top \mathbf{x})$.

Store the result in the list `ratios`.

In [ ]:
ratios = []

# ratio = time for (I + uv^T)x divided by time for x + u(v^T x), one entry per n
for compact_time, expanded_time in zip(compact_times, expanded_times):
    ratios.append(compact_time / expanded_time)

ratios


Once you have done the above work, inspect the ratio of computation times by running the following cell:

In [ ]:
plt.plot(n_vals, ratios)

Compare the computation times by describing how the ratio of the two grows as $n$ gets larger. 
Explain this in terms of the asymptotic temporal complexity of the two computations.

**Explanation**: The ratio doubles each time $n$ goes up by one.

With $N = 2^n$:

- $(I + \mathbf{u}\mathbf{v}^\top)\mathbf{x}$: building the $N \times N$ matrix and multiplying by $\mathbf{x}$ costs about $4N^2$ flops.
- $\mathbf{x} + \mathbf{u}(\mathbf{v}^\top \mathbf{x})$: only vector operations, about $4N$ flops.

The ratio of the leading terms is
$$\frac{4N^2}{4N} = N = 2^n,$$

so it doubles when $n$ increases by one. The first form is quadratic and the second is linear, so the gap keeps growing.

The plot agrees. The early values are noise, but from about $n = 8$ on the ratio climbs into the hundreds. The first form also has to store a whole $N \times N$ matrix, while the second only uses vectors. That is why the identity is worth using.

---

IMPORTANT: Please "Restart and Run All" and ensure there are no errors. Then, submit this .ipynb file to Gradescope.